# 3번 vs 11번 전처리 — SHAP 기반 요인 해석

**목표:** 3번(결측 제거 없음, 연속형 중앙값)과 11번(연속형 결측 평균 대체)을 선택했을 때,
각 모델(XGBoost 등)의 **SHAP 분석**으로 어떤 요인(변수)이 **경제활동 여부(1=참여, 0=비참여)** 예측에 어떻게 기여하는지 해석합니다.

- **양(+) 경향:** 해당 변수 값이 클수록 모델이 **경제활동(1)** 쪽으로 더 예측하는 경향
- **음(-) 경향:** 해당 변수 값이 클수록 **비경제활동(0)** 쪽으로 더 예측하는 경향

**사전 조건:** 3번·11번 폴더의 `results/new_XGBoost.pkl` (및 원하면 new_LightGBM, new_CatBoost)이 있어야 합니다.

## 1. 설정 및 공통 함수

In [ ]:
import os
import sys
import pickle
import numpy as np
import pandas as pd
from pandas import DataFrame
import shap

try:
    from IPython.display import display
except ImportError:
    display = print

PROJECT_ROOT = os.path.abspath(os.getcwd())
_fallback = r'C:\itwill_bigdata_final_project-main\itwill_bigdata_final_project'
if os.path.isdir(_fallback) and not os.path.isdir(os.path.join(PROJECT_ROOT, '3. 결측 변수 제거 없이 분석 진행')):
    PROJECT_ROOT = _fallback

def compute_shap_summary(estimator, x_train, preprocess_step_name='preprocess', model_step_name='model', max_display=30):
    """
    Pipeline( preprocess + model ) 과 원시 x_train으로 SHAP 요약 테이블 생성.
    estimator: fitted sklearn Pipeline with steps [preprocess, model].
    """
    preprocess = estimator.named_steps[preprocess_step_name]
    inner = estimator.named_steps[model_step_name]
    X_tr = preprocess.transform(x_train)
    if hasattr(X_tr, 'toarray'):
        X_tr = X_tr.toarray()
    feature_names = preprocess.get_feature_names_out()
    X_train_df = DataFrame(X_tr, columns=feature_names, index=x_train.index)

    explainer = shap.TreeExplainer(inner, data=X_train_df)
    shap_values = explainer.shap_values(X_train_df)
    if isinstance(shap_values, list):
        shap_values = shap_values[1]
    if hasattr(shap_values, 'ndim') and shap_values.ndim == 3:
        shap_values = shap_values[:, :, 1]

    shap_df = DataFrame(shap_values, columns=feature_names, index=x_train.index)
    total = shap_df.abs().mean().sum()
    summary_df = DataFrame({
        'feature': feature_names,
        'mean_abs_shap': shap_df.abs().mean().values,
        'mean_shap': shap_df.mean().values,
        'std_shap': shap_df.std().values,
    })
    summary_df['direction'] = np.where(
        summary_df['mean_shap'] > 0, '양(+) 경향',
        np.where(summary_df['mean_shap'] < 0, '음(-) 경향', '혼합/미약')
    )
    summary_df['importance_ratio'] = summary_df['mean_abs_shap'] / total
    summary_df = summary_df.sort_values('mean_abs_shap', ascending=False).reset_index(drop=True)
    return summary_df, shap_values, X_train_df

def interpret_top_features(summary_df, top_n=15, label=''):
    """상위 top_n개 요인에 대한 해석 문장 생성."""
    top = summary_df.head(top_n)
    lines = [f"**{label}** 상위 {top_n}개 요인:"]
    for i, row in top.iterrows():
        f = row['feature'].replace('num__', '').replace('cat__', '')
        d = row['direction']
        r = row['importance_ratio']
        if '양(+)' in str(d):
            meaning = '→ 경제활동(참여) 쪽으로 기여'
        elif '음(-)' in str(d):
            meaning = '→ 비경제활동 쪽으로 기여'
        else:
            meaning = '→ 방향 혼합/미약'
        lines.append(f"  {row.name+1}. {f} ({d}, 기여비 {r:.2%}) {meaning}")
    return '\n'.join(lines)

## 2. 3번 — 결측 제거 없음(연속형 중앙값) SHAP

In [ ]:
path_3 = os.path.join(PROJECT_ROOT, '3. 결측 변수 제거 없이 분석 진행', 'results', 'new_XGBoost.pkl')
if not os.path.isfile(path_3):
    print('3번 new_XGBoost.pkl 없음. 해당 폴더에서 new_XGBoost.ipynb 실행 후 다시 시도하세요.')
else:
    with open(path_3, 'rb') as f:
        data_3 = pickle.load(f)
    estimator_3 = data_3.get('estimator')
    x_train_3 = data_3.get('x_train')
    if estimator_3 is None or x_train_3 is None:
        print('3번 pkl에 estimator 또는 x_train 없음.')
    else:
        summary_3, shap_3, X_df_3 = compute_shap_summary(estimator_3, x_train_3)
        print('=== 3번 (결측 제거 없음, 연속형 중앙값) XGBoost SHAP 상위 30개 ===')
        display(summary_3.head(30))
        print()
        print(interpret_top_features(summary_3, top_n=15, label='3번 XGBoost'))

## 3. 11번 — 연속형 결측 평균 대체 SHAP

11번은 scenario_runner로 학습된 pipeline만 저장되어 있으므로, 동일한 데이터를 load_data()로 불러와 전처리 후 SHAP을 계산합니다.

In [ ]:
path_11 = os.path.join(PROJECT_ROOT, '11. 연속형결측_평균대체', 'results', 'new_XGBoost.pkl')
if not os.path.isfile(path_11):
    print('11번 new_XGBoost.pkl 없음. 해당 폴더에서 new_XGBoost.ipynb 실행 후 다시 시도하세요.')
else:
    sys.path.insert(0, PROJECT_ROOT)
    from scenario_runner import load_data
    x_train_11, x_test_11, y_train_11, y_test_11, _, _ = load_data()

    with open(path_11, 'rb') as f:
        data_11 = pickle.load(f)
    pipe_11 = data_11['pipeline']

    summary_11, shap_11, X_df_11 = compute_shap_summary(pipe_11, x_train_11, preprocess_step_name='preprocess', model_step_name='model')
    print('=== 11번 (연속형 결측 평균 대체) XGBoost SHAP 상위 30개 ===')
    display(summary_11.head(30))
    print()
    print(interpret_top_features(summary_11, top_n=15, label='11번 XGBoost'))

## 4. 3번 vs 11번 요약 비교 및 결과 해석

In [ ]:
try:
    top3 = set(summary_3.head(15)['feature'].tolist())
    top11 = set(summary_11.head(15)['feature'].tolist())
    common = top3 & top11
    only3 = top3 - top11
    only11 = top11 - top3
    print('=== 3번 vs 11번 상위 15개 요인 비교 ===')
    print(f'공통 상위 15: {len(common)}개')
    print(f'3번에만 상위 15: {len(only3)}개', list(only3)[:5], '...' if len(only3)>5 else '')
    print(f'11번에만 상위 15: {len(only11)}개', list(only11)[:5], '...' if len(only11)>5 else '')
    print('\n동일 변수의 방향(양/음) 일치 여부는 위 두 SHAP 표의 direction 컬럼을 비교하면 됩니다.')
except NameError:
    print('위에서 3번·11번 SHAP 셀을 먼저 실행한 뒤 다시 실행하세요.')

## 5. (선택) 3번·11번 다른 트리 모델 — LightGBM, CatBoost

동일 방식으로 LightGBM·CatBoost의 SHAP 상위 요인을 볼 수 있습니다. 아래에서 모델명만 바꿔 실행하세요.

In [ ]:
for folder_name, folder_key in [('3. 결측 변수 제거 없이 분석 진행', '3번'), ('11. 연속형결측_평균대체', '11번')]:
    res_dir = os.path.join(PROJECT_ROOT, folder_name, 'results')
    for model_name in ['new_LightGBM', 'new_CatBoost']:
        path = os.path.join(res_dir, model_name + '.pkl')
        if not os.path.isfile(path):
            continue
        with open(path, 'rb') as f:
            data = pickle.load(f)
        pipe = data.get('estimator') or data.get('pipeline')
        if pipe is None:
            continue
        if folder_key == '11번':
            if 'x_train_11' not in dir():
                sys.path.insert(0, PROJECT_ROOT)
                from scenario_runner import load_data
                x_train_11, _, _, _, _, _ = load_data()
            x_tr = x_train_11
        else:
            x_tr = data.get('x_train')
            if x_tr is None:
                continue
        try:
            summary, _, _ = compute_shap_summary(pipe, x_tr)
            print(f'=== {folder_key} {model_name.replace("new_", "")} SHAP 상위 10 ===')
            display(summary.head(10))
        except Exception as e:
            print(f'{folder_key} {model_name}: SHAP 계산 실패 -', e)

## 6. 보고서용 결과 해석 (요약)

**공통 해석 가이드:**
- **종속변수:** 경제활동 여부 (1=참여, 0=비참여)
- **mean_abs_shap**이 클수록 해당 변수의 예측 기여도가 크다.
- **양(+) 경향:** 변수 값이 높을수록 모델이 **경제활동 참여(1)** 로 예측하는 데 기여.
- **음(-) 경향:** 변수 값이 높을수록 **비경제활동(0)** 으로 예측하는 데 기여.

**변수명 참고 (일부):** `w09earned` 근로총소득, `w09a002_age` 연령, `w09ecoact_s` 배우자 경제활동 상태, `w09pinc` 개인총소득, `w09hhinc` 가구총소득, `w09c001` 건강상태, `num__`/`cat__` 접두사는 전처리 파이프라인 구분용.

**3번 선택 시:** 위 "3번 SHAP 상위 30개" 표를 인용해, 예를 들어 "근로총소득(w09earned), 연령(w09a002_age), 배우자 경제활동 상태(w09ecoact_s) 등이 상위 요인으로 나타났으며, …" 형태로 서술.

**11번 선택 시:** 위 "11번 SHAP 상위 30개" 표를 인용해 동일하게 서술. 3번과 11번은 연속형 결측 처리만 다르므로 요인 순위나 방향이 비슷할 수 있으나, 표로 제시한 뒤 "전처리 방법에 따라 일부 변수 중요도 순위가 달라질 수 있음"을 언급할 수 있음.